In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
url = 'https://raw.githubusercontent.com/ABahadirAvcibas/MachineLearning-ScikitLearn/refs/heads/main/Penguin-Classification/penguins_size.csv'

In [3]:
rawData = pd.read_csv(url)

In [4]:
rawData.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [5]:
rawData.shape

(344, 7)

In [6]:
rawData.isnull().sum()

,0
species,0
island,0
culmen_length_mm,2
culmen_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,10


In [7]:
rawData["species"].value_counts()

,count
species,
Adelie,152
Gentoo,124
Chinstrap,68


In [8]:
rawData

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


In [9]:
rawData["sex"].value_counts()

,count
sex,
MALE,168
FEMALE,165
.,1


In [10]:
## Since we have a '.' in sex column we are gonna make it NaN first

In [11]:
rawData["sex"] = rawData["sex"].replace(".", np.nan)

In [12]:
rawData["sex"].value_counts() # Now we don't have any '.' value

,count
sex,
MALE,168
FEMALE,165


In [13]:
rawData["sex"] = rawData["sex"].fillna(rawData["sex"].mode()[0])  # Filling up the sex column with the mode value

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
x = rawData.drop("species", axis=1)
y = rawData["species"]

In [16]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=94)

In [17]:
rawData["sex"].value_counts() # Since males were common compared to females, we filled up every missing value with male

,count
sex,
MALE,179
FEMALE,165


In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
le = LabelEncoder() # We are gonna apply label encoder to the 'sex' column  LABEL ENCODER = 0/1, ONE HOT ENCODER = 0,1,2

In [20]:
x_train["sex"] = le.fit_transform(x_train["sex"])
x_test["sex"] = le.transform(x_test["sex"])       # With those lines we just make sex either 0 or 1 for train and test variables

In [21]:
# culmen_length_mm	culmen_depth_mm	 flipper_length_mm	 body_mass_g - now we will fill the missing values for numerical datas

In [22]:
from sklearn.impute import SimpleImputer

In [23]:
imputer = SimpleImputer(strategy="mean")

In [24]:
numerical_columns = ["culmen_length_mm", "culmen_depth_mm",	"flipper_length_mm"	,"body_mass_g"]

In [25]:
x_train[numerical_columns] = imputer.fit_transform(x_train[numerical_columns])

In [26]:
x_test[numerical_columns] = imputer.transform(x_test[numerical_columns])

In [27]:
x_test.isnull().sum() # Both x_test and x_train values are now filled

,0
island,0
culmen_length_mm,0
culmen_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


In [28]:
# Adalar (Biscoe, Dream, Torgersen) arasında "Biscoe, Dream'den daha büyüktür" gibi bir hiyerarşi olmadığı için, onları One-Hot Encoding ile her ada için ayrı bir sütun
# açarak kodlamak en sağlıklısıdır.
# Since no island has a priority among each other, we are gonna apply OHE



In [29]:
from sklearn.preprocessing import OneHotEncoder

In [30]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [31]:
island_train_encoded = ohe.fit_transform(x_train[["island"]])
island_test_encoded = ohe.transform(x_test[["island"]])

In [32]:
column_names = ohe.get_feature_names_out(["island"])

In [33]:
island_train_df = pd.DataFrame(island_train_encoded, columns=column_names, index=x_train.index)
island_test_df = pd.DataFrame(island_test_encoded, columns=column_names, index=x_test.index)

In [34]:
x_train = pd.concat([x_train.drop('island', axis=1), island_train_df], axis=1)
x_test = pd.concat([x_test.drop('island', axis=1), island_test_df], axis=1)

In [35]:
from sklearn.preprocessing import StandardScaler

In [36]:
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [37]:
## KORELASYON ANALİZİ

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [39]:
model = LogisticRegression(random_state=94)

In [40]:
from sklearn.model_selection import cross_val_score

model.fit(x_train, y_train)

# CROSS VALIDATION
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"Cross-Validation Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Özelliklerin önemine bakalım.
# Eğer bir katsayı aşırı yüksekse, o özellik hedef değişkeni doğrudan sızdırıyor olabilir.
feature_importance = pd.DataFrame(model.coef_, columns=x_train if 'x_train_cols' in locals() else [f'Feature {i}' for i in range(x_train.shape[1])])
print("\nModel Coefficients (first class):\n", feature_importance.iloc[0])

Cross-Validation Accuracy: 0.9927 (+/- 0.0178)

Model Coefficients (first class):
 Feature 0   -2.347195
Feature 1    0.923896
Feature 2   -0.580998
Feature 3   -0.099464
Feature 4    0.553303
Feature 5   -0.068204
Feature 6   -0.273230
Feature 7    0.465490
Name: 0, dtype: float64


In [41]:
y_pred = model.predict(x_test)

In [42]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9855072463768116
